In [1]:
import pandas as pd
import numpy as np
import optuna
from darts import TimeSeries
from darts.models import AutoARIMA
from darts.metrics import mae
from darts.utils.missing_values import fill_missing_values
from darts.utils.utils import train_test_split
import matplotlib.pyplot as plt

SEED = 1234

# 1. Cargar y preparar los datos
datos = pd.read_csv("data/Temp_Asu20092021.csv")

# Convertir 'Fecha' a tipo datetime y establecerla como índice
datos['Fecha'] = pd.to_datetime(datos['Fecha'])
datos.set_index('Fecha', inplace=True)

# Filtrar datos desde 2019
datos = datos[datos.index.year >= 2019]

# Eliminar valores faltantes
datos = fill_missing_values(datos)  # Función de Darts para imputación de valores

# Calcular percentiles diarios
percentiles_diarios = datos.resample('D')['Temperatura'].agg(
    Percentil_95=lambda x: np.percentile(x, 95),
    Mediana=lambda x: np.percentile(x, 50),
    Percentil_5=lambda x: np.percentile(x, 5)
)




SyntaxError: invalid syntax (1421320755.py, line 1)

In [ ]:

# Crear las variables objetivo (día siguiente)
percentiles_diarios['Mediana_siguiente'] = percentiles_diarios['Mediana'].shift(-1)
percentiles_diarios.dropna(inplace=True)

# Crear objeto TimeSeries para Darts
serie = TimeSeries.from_dataframe(percentiles_diarios, value_cols=['Mediana', 'Percentil_95', 'Percentil_5'])
target = TimeSeries.from_dataframe(percentiles_diarios, value_cols=['Mediana_siguiente'])

# Dividir en entrenamiento y prueba
serie_train, serie_test = train_test_split(serie, test_size=0.2, shuffle=False)
target_train, target_test = train_test_split(target, test_size=0.2, shuffle=False)

# 2. Definir la función objetivo para Optuna
def objetivo(trial):
    p = trial.suggest_int('p', 0, 5)
    d = trial.suggest_int('d', 0, 2)
    q = trial.suggest_int('q', 0, 5)
    P = trial.suggest_int('P', 0, 2)
    D = trial.suggest_int('D', 0, 1)
    Q = trial.suggest_int('Q', 0, 2)
    s = trial.suggest_int('s', 7, 30)

    # Crear modelo AutoARIMA con los hiperparámetros sugeridos
    modelo = AutoARIMA(start_p=p, start_q=q, d=d, seasonal=True,
                       start_P=P, start_Q=Q, D=D, m=s)

    # Ajustar el modelo
    modelo.fit(serie_train)

    # Predecir el horizonte de prueba
    predicciones = modelo.predict(len(target_test))

    # Calcular MAE
    error = mae(target_test, predicciones)
    return error

# 3. Optimización con Optuna
estudio = optuna.create_study(direction='minimize')
estudio.optimize(objetivo, n_trials=30)

# 4. Imprimir los mejores hiperparámetros
print("Mejores hiperparámetros:")
print(estudio.best_params)

# 5. Ajustar el modelo final con los mejores hiperparámetros
mejores_hyperparams = estudio.best_params
modelo_final = AutoARIMA(start_p=mejores_hyperparams['p'],
                         start_q=mejores_hyperparams['q'],
                         d=mejores_hyperparams['d'],
                         seasonal=True,
                         start_P=mejores_hyperparams['P'],
                         start_Q=mejores_hyperparams['Q'],
                         D=mejores_hyperparams['D'],
                         m=mejores_hyperparams['s'])

modelo_final.fit(serie_train)

# 6. Evaluar el modelo final
predicciones_finales = modelo_final.predict(len(target_test))
mae_final = mae(target_test, predicciones_finales)
print(f'MAE en validación: {mae_final:.2f}')

# 7. Visualización
plt.figure(figsize=(10, 6))
target_test.plot(label='Observado')
predicciones_finales.plot(label='Predicción', linestyle='--')
plt.legend()
plt.show()

In [ ]:


# 7. Graficar predicciones
plt.figure(figsize=(10, 5))
plt.plot(y_test, label='Valor Real', color='blue')
plt.plot(predicciones_finales, label='Predicción SARIMAX', color='red', linestyle='--')
plt.xlabel('Días')
plt.ylabel('Mediana de Temperatura')
plt.title('Predicción de la Mediana de Temperatura con SARIMAX Optimizado')
plt.legend()
plt.show()

[I 2024-10-24 08:28:57,619] A new study created in memory with name: no-name-ce1ae859-55ad-4e29-8d3d-619cd5d59b86
